In [1]:
import os
import time
import random
import requests
import pandas as pd
from urllib.parse import urljoin
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- Configuration --------------------
BASE_URL = "https://www.finlex.fi"
INDEX_URL = f"{BASE_URL}/en/authorities/collective-agreements/"
OUTPUT_DIR = "finlex_az_data"
PDF_DIR = os.path.join(OUTPUT_DIR, "pdfs")
CSV_PATH = os.path.join(OUTPUT_DIR, "finlex_az_detailed_metadata.csv")

def make_driver():
    opts = Options()
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--disable-dev-shm-usage") 
    
    # Run in Headless Mode
    #opts.add_argument("--headless=new") 
    
    # Spoof User-Agent so Finlex doesn't block the headless browser
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    opts.add_argument(f"user-agent={user_agent}")
    
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

def download_pdf(pdf_url, save_dir):
    """Downloads a PDF with Exponential Backoff for 429 blocks."""
    try:
        raw_filename = pdf_url.split('/')[-1]
        clean_filename = raw_filename.split('?')[0]
        filepath = os.path.join(save_dir, clean_filename)
        
        if os.path.exists(filepath):
            return clean_filename
            
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
        max_retries = 4
        for attempt in range(max_retries):
            time.sleep(random.uniform(1.0, 2.5)) # Humanized delay
            response = requests.get(pdf_url, headers=headers, stream=True, timeout=20)
            
            if response.status_code == 200:
                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                return clean_filename
                
            elif response.status_code == 429:
                sleep_time = (15 * (2 ** attempt)) + random.uniform(1, 5)
                print(f"      [!] Rate limited (429). Sleeping for {sleep_time:.1f}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(sleep_time)
            else:
                print(f"      [!] Failed to download. HTTP Status: {response.status_code}")
                return None
                
        print(f"      [!] Failed to download PDF after {max_retries} attempts.")
        return None 
        
    except Exception as e:
        print(f"      [!] Exception downloading PDF: {e}")
        return None

def gather_all_agreement_urls(d):
    """Phase 1: Navigate A-Z and handle hardcoded pagination for A, M, T."""
    print(f"\n{'='*60}")
    print(" PHASE 1: Harvesting all agreement URLs via A-Z Index")
    print(f"{'='*60}")
    
    d.get(INDEX_URL)
    time.sleep(3)
    
    # Find all the A-Z buttons
    alphabet_urls = []
    for a in d.find_elements(By.TAG_NAME, "a"):
        text = a.text.strip()
        if len(text) == 1 and text.isalpha() and text.isupper():
            href = a.get_attribute("href")
            if href and href not in alphabet_urls:
                alphabet_urls.append(href)
                
    print(f"Detected {len(alphabet_urls)} alphabetical letter filters.")
    
    all_agreement_urls = set()
    
    for letter_url in alphabet_urls:
        letter = letter_url.split('/')[-1].upper()
        print(f"\nScanning Letter: {letter}")
        d.get(letter_url)
        time.sleep(random.uniform(2.0, 4.0))
        
        # --- Scrape Page 1 ---
        links = d.find_elements(By.CSS_SELECTOR, "li[class*='documentListItem'] a")
        page1_count = 0
        for link in links:
            href = link.get_attribute("href")
            if href:
                all_agreement_urls.add(href)
                page1_count += 1
                
        print(f"  -> Page 1: Found {page1_count} agreements.")
        
        # --- Handle Page 2 for specific letters ---
        if letter in ['A', 'M', 'T']:
            try:
                print(f"  -> Letter {letter} has multiple pages. Attempting to load Page 2...")
                
                # Look for the specific aria-label you provided
                page2_btn = WebDriverWait(d, 5).until(
                    EC.presence_of_element_located((By.XPATH, "//a[@aria-label='Page 2']"))
                )
                
                # Scroll it into view and use Javascript to click (bypasses sticky headers/banners)
                d.execute_script("arguments[0].scrollIntoView(true);", page2_btn)
                time.sleep(1)
                d.execute_script("arguments[0].click();", page2_btn)
                
                # Wait for the DOM to update with new links
                time.sleep(random.uniform(3.0, 5.0))
                
                # Scrape Page 2
                links_p2 = d.find_elements(By.CSS_SELECTOR, "li[class*='documentListItem'] a")
                page2_count = 0
                for link in links_p2:
                    href = link.get_attribute("href")
                    # Make sure we don't count duplicates if the page didn't actually change
                    if href and href not in all_agreement_urls: 
                        all_agreement_urls.add(href)
                        page2_count += 1
                        
                print(f"  -> Page 2: Found {page2_count} additional agreements.")
                
            except Exception as e:
                print(f"  -> [!] Could not navigate to Page 2 for {letter}. Error: {e}")

    print(f"\nPhase 1 Complete! Successfully harvested {len(all_agreement_urls)} total unique agreement URLs.")
    return list(all_agreement_urls)

def scrape_agreements():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(PDF_DIR, exist_ok=True)
    
    d = make_driver()
    
    try:
        # Phase 1: Get Master List of URLs
        master_urls = gather_all_agreement_urls(d)
        
        # Phase 2: Deep Scrape
        print(f"\n{'='*60}")
        print(" PHASE 2: Deep Scraping Metadata and PDFs")
        print(f"{'='*60}")
        
        all_metadata = []
        
        for i, url in enumerate(master_urls):
            print(f"[{i+1}/{len(master_urls)}] Navigating to: {url}")
            
            # Setup base data
            agreement_type = "Central Organisation Agreement" if "trade-union-center-agreements" in url else "General Applicability Agreement"
            
            # Attempt to extract year from URL if possible
            year = url.split('/')[-2] if url.split('/')[-2].isdigit() else "Unknown"
            
            doc_data = {"Year": year, "Agreement_Type": agreement_type, "Source_URL": url}
            
            # Crash Recovery Loop
            success = False
            for attempt in range(3):
                try:
                    # Test if driver crashed. If it did, rebuild it.
                    try:
                        d.title 
                    except WebDriverException:
                        print("   [!] Browser crashed. Rebuilding driver...")
                        d = make_driver()
                        
                    time.sleep(random.uniform(2.5, 5.0))
                    d.get(url)
                    
                    WebDriverWait(d, 15).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "dl[class*='listContainer']"))
                    )
                    
                    try:
                        title_elem = d.find_element(By.CSS_SELECTOR, "div[class*='titleContainer'] h2")
                        doc_data["Main_Title"] = title_elem.text.strip()
                    except:
                        doc_data["Main_Title"] = "N/A"
                    
                    # Parse Table Data
                    dts = d.find_elements(By.TAG_NAME, "dt")
                    dds = d.find_elements(By.TAG_NAME, "dd")
                    for dt, dd in zip(dts, dds):
                        key = dt.text.strip()
                        if key.lower() == "documents": continue
                        doc_data[key] = dd.text.strip()
                    
                    # Parse and Download PDFs
                    pdf_links = d.find_elements(By.CSS_SELECTOR, "a[class*='pdfLink']")
                    downloaded_files = []
                    
                    if pdf_links:
                        print(f"   Found {len(pdf_links)} PDF(s). Downloading...")
                        for pdf_a in pdf_links:
                            pdf_href = pdf_a.get_attribute("href")
                            full_pdf_url = urljoin(BASE_URL, pdf_href) 
                            saved_filename = download_pdf(full_pdf_url, PDF_DIR)
                            if saved_filename:
                                downloaded_files.append(saved_filename)
                                print(f"      -> Saved: {saved_filename}")
                    else:
                        print("   No PDFs found.")
                    
                    # Store PDFs in distinct columns
                    for j, pdf_file in enumerate(downloaded_files):
                        doc_data[f"PDF_{j+1}"] = pdf_file
                        
                    all_metadata.append(doc_data)
                    success = True
                    break 
                    
                except Exception as e:
                    print(f"   [!] Error extracting data (Attempt {attempt+1}/3): {e}")
                    time.sleep(random.uniform(5.0, 10.0))
            
            if not success:
                print(f"   [!] Failed to scrape {url} entirely after 3 attempts.")

            # Incremental Save every 10 records to avoid data loss
            if (i + 1) % 10 == 0 or (i + 1) == len(master_urls):
                df = pd.DataFrame(all_metadata)
                cols = df.columns.tolist()
                priority_cols = ["Year", "Agreement_Type", "Source_URL", "Main_Title"]
                other_cols = [c for c in cols if c not in priority_cols]
                df = df[priority_cols + other_cols]
                
                df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
                print(f"-> Progress auto-saved. Currently {len(df)} records in CSV.\n")

    finally:
        try: d.quit() 
        except: pass

    print(f"\nFINISHED! All data saved to: {CSV_PATH}")
    print(f"PDFs saved to: {os.path.abspath(PDF_DIR)}")

if __name__ == "__main__":
    scrape_agreements()


 PHASE 1: Harvesting all agreement URLs via A-Z Index
Detected 22 alphabetical letter filters.

Scanning Letter: A
  -> Page 1: Found 20 agreements.
  -> Letter A has multiple pages. Attempting to load Page 2...
  -> Page 2: Found 4 additional agreements.

Scanning Letter: B
  -> Page 1: Found 2 agreements.

Scanning Letter: E
  -> Page 1: Found 9 agreements.

Scanning Letter: F
  -> Page 1: Found 1 agreements.

Scanning Letter: G
  -> Page 1: Found 1 agreements.

Scanning Letter: H
  -> Page 1: Found 14 agreements.

Scanning Letter: I
  -> Page 1: Found 3 agreements.

Scanning Letter: J
  -> Page 1: Found 1 agreements.

Scanning Letter: K
  -> Page 1: Found 18 agreements.

Scanning Letter: L
  -> Page 1: Found 13 agreements.

Scanning Letter: M
  -> Page 1: Found 20 agreements.
  -> Letter M has multiple pages. Attempting to load Page 2...
  -> Page 2: Found 11 additional agreements.

Scanning Letter: N
  -> Page 1: Found 2 agreements.

Scanning Letter: O
  -> Page 1: Found 3 agreeme